# Notebook 4: Linear Probing
**Kurs:** Mechanistic Interpretability  
**Modell:** EleutherAI/pythia-410m  
**Ziel:** Lineare Probes trainieren, um zu testen, welche Information in welcher Schicht kodiert ist.

> **Aufgabe:** Implementiere die Zellen schrittweise. Hilfreiche Dokumentation:
> - PyTorch Hooks: https://pytorch.org/docs/stable/generated/torch.nn.Module.register_forward_hook.html
> - Scikit-Learn LogisticRegression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
> - Scikit-Learn cross_val_score: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html
> - MLflow: https://mlflow.org/docs/latest/python_api/mlflow.html

## Hintergrund: Was ist lineares Probing?

**Probing** testet, ob bestimmte Information in den Aktivierungen einer Schicht **linear dekodierbar** ist.

**Vorgehen:**
1. Erstelle Datensatz mit Labels (korrekte vs. falsche Hauptstadt)
2. Extrahiere Hidden States jeder Schicht mit **Forward Hooks**
3. Trainiere **LogisticRegression** (Probe) pro Schicht
4. Plotte Genauigkeit pro Schicht → "Informationsbildungskurve"

**Warum Forward Hooks?**
Forward Hooks sind PyTorch-Callbacks, die nach jedem Forward Pass durch ein Modul aufgerufen werden. So können wir Aktivierungen "abgreifen", ohne das Modell zu verändern.

```python
def hook_fn(module, input, output):
    # output[0] enthält die Hidden States dieser Schicht
    aktivierungen[layer_idx] = output[0].detach().cpu()

layer.register_forward_hook(hook_fn)
```

**Warum linear?** Ein linearer Probe ist zu schwach, um selbst komplexe Muster zu lernen — wenn er trotzdem gut funktioniert, liegt es am Modell.

## 1. Setup

In [ ]:
# TODO: Importiere torch, numpy, matplotlib
# TODO: Importiere LogisticRegression und cross_val_score aus sklearn
# TODO: Importiere StandardScaler aus sklearn.preprocessing
# TODO: Lade Tokenizer und Modell (wie in Notebook 1)
#
# Optional: Versuche mlflow zu importieren (try/except)
# MLflow-Dokumentation: https://mlflow.org/docs/latest/python_api/mlflow.html

## 2. Datensatz erstellen

In [ ]:
# TODO: Erstelle einen Datensatz mit mindestens 20 Länder-Hauptstadt-Paaren.
# Für jedes Paar erstelle TWO Prompts:
#   Label 1 (korrekt): "The capital of <Country> is <CorrectCapital>"
#   Label 0 (falsch):  "The capital of <Country> is <WrongCapital>"
#
# Beispiel-Struktur:
# COUNTRY_CAPITAL_PAIRS = [
#     ("France", "Paris", "Madrid"),
#     ("Germany", "Berlin", "Vienna"),
#     ...
# ]
#
# Hinweis: Du brauchst mindestens 20 Paare = 40 Beispiele für sinnvolle Kreuzvalidierung
# Hinweis: Die "falsche" Hauptstadt sollte eine echte Hauptstadt sein (kein Nonsense)
#
# Ausgabe: all_prompts (Liste aller Prompts), labels (np.array mit 0/1)

## 3. Aktivierungen mit Forward Hooks extrahieren

In [ ]:
# TODO: Implementiere extract_all_layer_activations(prompts, batch_size=4):
#
# Schritt 1: Registriere einen Forward Hook auf JEDER Schicht:
#   for i, layer in enumerate(model.gpt_neox.layers):
#       hooks.append(layer.register_forward_hook(make_hook(i)))
#
# Schritt 2: Führe einen Forward Pass durch und sammle die Aktivierungen.
#   WICHTIG: Nutze output[0] (der erste Eintrag des Outputs ist der Hidden State)
#
# Schritt 3: Entferne alle Hooks nach dem Forward Pass:
#   for h in hooks: h.remove()
#
# Schritt 4: Extrahiere den Hidden State am LETZTEN gültigen Token:
#   - Nutze attention_mask um den letzten nicht-Padding-Token zu finden
#   - last_valid = attention_mask[sample_idx].sum().item() - 1
#
# Dokumentation zu register_forward_hook:
#   https://pytorch.org/docs/stable/generated/torch.nn.Module.register_forward_hook.html
#
# WICHTIG: Verwende torch.no_grad() für alle Forward Passes\!

## 4. Lineare Probes trainieren

In [ ]:
# TODO: Implementiere train_probe_per_layer(activations, labels, n_layers):
#
# Für JEDE Schicht:
#   1. X = activations[layer_idx]   # Shape: (n_samples, hidden_size)
#   2. Standardisiere X mit StandardScaler (wichtig für Logistische Regression\!)
#      scaler = StandardScaler()
#      X_scaled = scaler.fit_transform(X)
#   3. Erstelle LogisticRegression(max_iter=1000, C=1.0)
#   4. Berechne 5-fache Kreuzvalidierung:
#      cv_scores = cross_val_score(probe, X_scaled, y, cv=5, scoring="accuracy")
#   5. Speichere Mittelwert und Standardabweichung
#
# Rückgabe: accuracies (n_layers,), std_devs (n_layers,)
#
# Tipp: StandardScaler-Dokumentation:
#   https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

## 5. Informationsbildungskurve plotten

In [ ]:
# TODO: Erstelle einen Linienchart mit plt.plot():
#   - X-Achse: Schicht-Index
#   - Y-Achse: CV-Genauigkeit
#   - Fehlerband: plt.fill_between(..., acc - std, acc + std, alpha=0.2)
#   - Horizontale Linie bei 0.5 (Chance-Level): plt.axhline(y=0.5, ...)
#
# TODO: Annotiere die Schicht mit der besten Genauigkeit:
#   best_layer = int(accuracies.argmax())
#   plt.annotate(f"Max: {accuracies[best_layer]:.3f}", ...)
#
# Frage: In welcher Schicht bildet sich das Wissen über die Hauptstadt heraus?
#
# Dokumentation plt.fill_between:
#   https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.fill_between.html

## 6. MLflow Experiment-Tracking

In [ ]:
# TODO: Falls MLflow verfügbar, logge das Experiment:
#
# mlflow.set_experiment("linear_probing_pythia410m")
# with mlflow.start_run(run_name="capital_city_probe"):
#     mlflow.log_param("model", MODEL_NAME)
#     mlflow.log_param("n_examples", len(all_prompts))
#     for layer_idx, acc in enumerate(accuracies):
#         mlflow.log_metric(f"layer_{layer_idx:02d}_accuracy", acc, step=layer_idx)
#     mlflow.log_metric("best_accuracy", accuracies.max())
#     mlflow.log_metric("best_layer", int(accuracies.argmax()))
#
# WICHTIG: Falls MLflow nicht installiert ist, gib die Ergebnisse als print()-Tabelle aus.
#
# MLflow-Dokumentation: https://mlflow.org/docs/latest/python_api/mlflow.html
# MLflow starten: mlflow ui  (im Terminal)

## 7. Reflexionsfragen

In [ ]:
# Beantworte folgende Fragen:
#
# 1. In welcher Schicht erreicht die Probe-Genauigkeit ihr Maximum?
# 2. Was bedeutet eine Genauigkeit von ~50%? Was bedeutet ~90%?
# 3. Was ist der Unterschied zwischen Probing und Activation Patching?
#    (Denke an: Korrelation vs. Kausalität)
# 4. Warum ist Standardisierung (StandardScaler) vor dem Probe-Training wichtig?
# 5. Was würde passieren, wenn du einen nicht-linearen Probe (z.B. MLP) verwendest?
#    Wäre das besser oder schlechter für Interpretierbarkeit? Warum?